# 1. MongoDB CRUD Operations

CRUD operations in MongoDB using PyMongo:

| Operation | Method | Description |
|-----------|--------|-------------|
| **C**reate | `insert_one()`, `insert_many()` | Add documents |
| **R**ead | `find_one()`, `find()` | Query documents |
| **U**pdate | `update_one()`, `update_many()` | Modify documents |
| **D**elete | `delete_one()`, `delete_many()` | Remove documents |

# 2. Setup

In [ ]:
from pymongo import MongoClient
from bson.objectid import ObjectId
from datetime import datetime
import pprint

# Connect to MongoDB
client = MongoClient("mongodb://localhost:27017/")
db = client["crud_demo"]
users = db["users"]

# Clear collection for fresh start
users.delete_many({})

# Pretty print function
def show(cursor_or_doc, title=""):
    if title:
        print(f"\n{title}")
        print("=" * 50)
    if hasattr(cursor_or_doc, '__iter__') and not isinstance(cursor_or_doc, dict):
        for doc in cursor_or_doc:
            pprint.pprint(doc)
    else:
        pprint.pprint(cursor_or_doc)

print("Setup complete! Database: crud_demo, Collection: users")

# 3. CREATE - Insert Operations

## 3.1 Insert Single Document

In [ ]:
# Insert a single document
user = {
    "name": "John Doe",
    "email": "john@email.com",
    "age": 30,
    "active": True,
    "tags": ["developer", "python"],
    "address": {
        "street": "123 Main St",
        "city": "New York",
        "zip": "10001"
    },
    "created_at": datetime.now()
}

result = users.insert_one(user)

print(f"Inserted document ID: {result.inserted_id}")
print(f"Acknowledged: {result.acknowledged}")

## 3.2 Insert Multiple Documents

In [ ]:
# Insert multiple documents
many_users = [
    {
        "name": "Jane Smith",
        "email": "jane@email.com",
        "age": 25,
        "active": True,
        "tags": ["designer", "ui"],
        "address": {"city": "Boston"},
        "created_at": datetime.now()
    },
    {
        "name": "Bob Wilson",
        "email": "bob@email.com",
        "age": 35,
        "active": False,
        "tags": ["manager"],
        "address": {"city": "Chicago"},
        "created_at": datetime.now()
    },
    {
        "name": "Alice Brown",
        "email": "alice@email.com",
        "age": 28,
        "active": True,
        "tags": ["developer", "javascript"],
        "address": {"city": "New York"},
        "salary": 75000,  # Different fields are OK!
        "created_at": datetime.now()
    },
    {
        "name": "Charlie Davis",
        "email": "charlie@email.com",
        "age": 40,
        "active": True,
        "tags": ["developer", "python", "data"],
        "address": {"city": "Seattle"},
        "created_at": datetime.now()
    }
]

result = users.insert_many(many_users)

print(f"Inserted {len(result.inserted_ids)} documents")
print(f"IDs: {result.inserted_ids}")

## 3.3 Insert with Custom _id

In [ ]:
# You can provide your own _id
custom_user = {
    "_id": "user_eva_001",  # Custom string ID
    "name": "Eva Martinez",
    "email": "eva@email.com",
    "age": 32
}

try:
    result = users.insert_one(custom_user)
    print(f"Inserted with custom ID: {result.inserted_id}")
except Exception as e:
    print(f"Error (probably duplicate _id): {e}")

# 4. READ - Query Operations

## 4.1 Find One Document

In [ ]:
# Find one document (returns first match)
user = users.find_one({"name": "John Doe"})
show(user, "Find one by name")

# Find by _id (must use ObjectId for auto-generated IDs)
# user = users.find_one({"_id": ObjectId("507f1f77bcf86cd799439011")})

# Find with custom string _id
user = users.find_one({"_id": "user_eva_001"})
show(user, "Find by custom _id")

## 4.2 Find All Documents

In [ ]:
# Find all documents (returns cursor)
all_users = users.find()

print("All users:")
print("="*50)
for user in all_users:
    print(f"{user['name']} - {user['email']}")

# Count documents
count = users.count_documents({})
print(f"\nTotal users: {count}")

## 4.3 Query Operators

In [ ]:
# ==========================================
# COMPARISON OPERATORS
# ==========================================

# $eq - Equal (default, can omit)
result = users.find({"age": 30})  # Same as {"age": {"$eq": 30}}
show(list(result), "Age equals 30")

# $ne - Not equal
result = users.find({"age": {"$ne": 30}})
print(f"\nNot 30 years old: {[u['name'] for u in result]}")

# $gt, $gte - Greater than, Greater than or equal
result = users.find({"age": {"$gt": 30}})
print(f"Age > 30: {[u['name'] for u in result]}")

result = users.find({"age": {"$gte": 30}})
print(f"Age >= 30: {[u['name'] for u in result]}")

# $lt, $lte - Less than, Less than or equal
result = users.find({"age": {"$lt": 30}})
print(f"Age < 30: {[u['name'] for u in result]}")

# $in - Value in array
result = users.find({"age": {"$in": [25, 30, 35]}})
print(f"Age is 25, 30, or 35: {[u['name'] for u in result]}")

# $nin - Value not in array
result = users.find({"age": {"$nin": [25, 30, 35]}})
print(f"Age is NOT 25, 30, or 35: {[u['name'] for u in result]}")

In [ ]:
# ==========================================
# LOGICAL OPERATORS
# ==========================================

# $and - All conditions must match
result = users.find({
    "$and": [
        {"age": {"$gte": 25}},
        {"active": True}
    ]
})
print(f"Age >= 25 AND active: {[u['name'] for u in result]}")

# Implicit AND (just use multiple conditions)
result = users.find({"age": {"$gte": 25}, "active": True})
print(f"Same with implicit AND: {[u['name'] for u in result]}")

# $or - Any condition can match
result = users.find({
    "$or": [
        {"age": {"$lt": 28}},
        {"address.city": "New York"}
    ]
})
print(f"\nAge < 28 OR from New York: {[u['name'] for u in result]}")

# $not - Negates a condition
result = users.find({"age": {"$not": {"$gt": 30}}})
print(f"NOT age > 30: {[u['name'] for u in result]}")

# $nor - None of the conditions match
result = users.find({
    "$nor": [
        {"active": False},
        {"age": {"$lt": 28}}
    ]
})
print(f"NOR (active=False, age<28): {[u['name'] for u in result]}")

In [ ]:
# ==========================================
# ELEMENT OPERATORS
# ==========================================

# $exists - Field exists or not
result = users.find({"salary": {"$exists": True}})
print(f"Has salary field: {[u['name'] for u in result]}")

result = users.find({"salary": {"$exists": False}})
print(f"No salary field: {[u['name'] for u in result]}")

# $type - Field is of specific BSON type
result = users.find({"age": {"$type": "int"}})
print(f"\nAge is integer: {[u['name'] for u in result]}")

In [ ]:
# ==========================================
# ARRAY OPERATORS
# ==========================================

# Find where array contains value
result = users.find({"tags": "python"})
print(f"Has 'python' tag: {[u['name'] for u in result]}")

# $all - Array contains all values
result = users.find({"tags": {"$all": ["developer", "python"]}})
print(f"Has both 'developer' AND 'python': {[u['name'] for u in result]}")

# $size - Array is specific size
result = users.find({"tags": {"$size": 2}})
print(f"\nHas exactly 2 tags: {[u['name'] for u in result]}")

# $elemMatch - Element matches all conditions
# Useful for arrays of objects
# Example: db.orders.find({"items": {"$elemMatch": {"product": "laptop", "qty": {"$gt": 1}}}})

In [ ]:
# ==========================================
# TEXT SEARCH OPERATORS
# ==========================================

# $regex - Regular expression match
result = users.find({"name": {"$regex": "^J"}})
print(f"Name starts with 'J': {[u['name'] for u in result]}")

# Case insensitive regex
result = users.find({"email": {"$regex": "@email.com$", "$options": "i"}})
print(f"Email ends with @email.com: {[u['name'] for u in result]}")

# Alternative Python regex syntax
import re
result = users.find({"name": re.compile("^A", re.IGNORECASE)})
print(f"Name starts with 'A' (case-insensitive): {[u['name'] for u in result]}")

## 4.4 Projection (Selecting Fields)

In [ ]:
# Projection: Select which fields to return

# Include specific fields (1 = include)
result = users.find({}, {"name": 1, "email": 1})
show(list(result), "Only name and email (+ _id by default)")

# Exclude _id
result = users.find({}, {"name": 1, "email": 1, "_id": 0})
show(list(result), "Name and email without _id")

# Exclude specific fields (0 = exclude)
result = users.find({}, {"address": 0, "tags": 0, "created_at": 0})
show(list(result), "Exclude address, tags, created_at")

# Note: Can't mix include and exclude (except for _id)

## 4.5 Sort, Limit, Skip

In [ ]:
from pymongo import ASCENDING, DESCENDING

# Sort ascending (1 or ASCENDING)
result = users.find().sort("age", ASCENDING)
print("Sorted by age (ascending):")
for u in result:
    print(f"  {u['name']}: {u['age']}")

# Sort descending (-1 or DESCENDING)
result = users.find().sort("age", DESCENDING)
print("\nSorted by age (descending):")
for u in result:
    print(f"  {u['name']}: {u['age']}")

# Sort by multiple fields
result = users.find().sort([("active", DESCENDING), ("age", ASCENDING)])
print("\nSorted by active (desc), then age (asc):")
for u in result:
    print(f"  {u['name']}: active={u['active']}, age={u['age']}")

In [ ]:
# Limit - Restrict number of results
result = users.find().limit(3)
print(f"First 3 users: {[u['name'] for u in result]}")

# Skip - Skip first N results
result = users.find().skip(2)
print(f"Skip first 2: {[u['name'] for u in result]}")

# Pagination with skip and limit
page_size = 2
page_number = 2  # 0-indexed

result = users.find().skip(page_number * page_size).limit(page_size)
print(f"\nPage {page_number + 1} (size {page_size}): {[u['name'] for u in result]}")

# Chaining: sort + skip + limit
result = users.find().sort("age", DESCENDING).skip(1).limit(2)
print(f"\nTop 2-3 oldest: {[(u['name'], u['age']) for u in result]}")

## 4.6 Querying Nested Documents and Arrays

In [ ]:
# Query nested fields using dot notation
result = users.find({"address.city": "New York"})
print(f"From New York: {[u['name'] for u in result]}")

# Query nested with operators
result = users.find({"address.city": {"$in": ["New York", "Boston"]}})
print(f"From NY or Boston: {[u['name'] for u in result]}")

# Project nested fields
result = users.find({}, {"name": 1, "address.city": 1, "_id": 0})
show(list(result), "\nNames with cities only")

# 5. UPDATE Operations

## 5.1 Update Operators

| Operator | Description |
|----------|-------------|
| `$set` | Set field value |
| `$unset` | Remove field |
| `$inc` | Increment value |
| `$mul` | Multiply value |
| `$rename` | Rename field |
| `$push` | Add to array |
| `$pull` | Remove from array |
| `$addToSet` | Add unique to array |

In [ ]:
# Update One Document

# $set - Set/update a field
result = users.update_one(
    {"name": "John Doe"},
    {"$set": {"age": 31, "title": "Senior Developer"}}
)

print(f"Matched: {result.matched_count}, Modified: {result.modified_count}")

# Verify
user = users.find_one({"name": "John Doe"})
print(f"John's new age: {user['age']}, title: {user.get('title')}")

In [ ]:
# $inc - Increment a number
result = users.update_one(
    {"name": "John Doe"},
    {"$inc": {"age": 1}}  # Add 1 to age
)

user = users.find_one({"name": "John Doe"})
print(f"John's age after $inc: {user['age']}")

# $inc can also decrement (negative value)
result = users.update_one(
    {"name": "John Doe"},
    {"$inc": {"age": -2}}  # Subtract 2
)

user = users.find_one({"name": "John Doe"})
print(f"John's age after -2: {user['age']}")

In [ ]:
# $unset - Remove a field
result = users.update_one(
    {"name": "John Doe"},
    {"$unset": {"title": ""}}  # Value doesn't matter for $unset
)

user = users.find_one({"name": "John Doe"})
print(f"John has title field: {'title' in user}")

In [ ]:
# Array operations

# $push - Add to array
result = users.update_one(
    {"name": "John Doe"},
    {"$push": {"tags": "mongodb"}}
)
user = users.find_one({"name": "John Doe"})
print(f"Tags after $push: {user['tags']}")

# $addToSet - Add only if not exists
result = users.update_one(
    {"name": "John Doe"},
    {"$addToSet": {"tags": "mongodb"}}  # Won't add duplicate
)
user = users.find_one({"name": "John Doe"})
print(f"Tags after $addToSet (duplicate): {user['tags']}")

# $pull - Remove from array
result = users.update_one(
    {"name": "John Doe"},
    {"$pull": {"tags": "mongodb"}}
)
user = users.find_one({"name": "John Doe"})
print(f"Tags after $pull: {user['tags']}")

# $push multiple with $each
result = users.update_one(
    {"name": "John Doe"},
    {"$push": {"tags": {"$each": ["sql", "api"]}}}
)
user = users.find_one({"name": "John Doe"})
print(f"Tags after $each: {user['tags']}")

In [ ]:
# Update Many Documents

result = users.update_many(
    {"active": True},
    {"$set": {"status": "verified"}}
)

print(f"Matched: {result.matched_count}, Modified: {result.modified_count}")

# Verify
verified = users.find({"status": "verified"})
print(f"Verified users: {[u['name'] for u in verified]}")

In [ ]:
# Update nested fields

result = users.update_one(
    {"name": "John Doe"},
    {"$set": {"address.country": "USA", "address.zip": "10002"}}
)

user = users.find_one({"name": "John Doe"})
show(user['address'], "John's updated address")

In [ ]:
# Upsert - Update or Insert if not exists

result = users.update_one(
    {"email": "frank@email.com"},  # Doesn't exist
    {"$set": {"name": "Frank Wilson", "age": 45}},
    upsert=True  # Create if not found
)

print(f"Matched: {result.matched_count}")
print(f"Modified: {result.modified_count}")
print(f"Upserted ID: {result.upserted_id}")

# Verify
user = users.find_one({"email": "frank@email.com"})
show(user, "Frank (upserted)")

# 6. DELETE Operations

In [ ]:
# Delete One Document

result = users.delete_one({"email": "frank@email.com"})
print(f"Deleted count: {result.deleted_count}")

# Verify
user = users.find_one({"email": "frank@email.com"})
print(f"Frank exists: {user is not None}")

In [ ]:
# Delete Many Documents

# First, add some test documents
users.insert_many([
    {"name": "Temp1", "temp": True},
    {"name": "Temp2", "temp": True},
    {"name": "Temp3", "temp": True}
])

# Delete all temp users
result = users.delete_many({"temp": True})
print(f"Deleted count: {result.deleted_count}")

In [ ]:
# Find and Delete (returns deleted document)

# Add a user to delete
users.insert_one({"name": "To Be Deleted", "email": "delete@email.com"})

# Find and delete - returns the deleted document
deleted_user = users.find_one_and_delete({"email": "delete@email.com"})

if deleted_user:
    print(f"Deleted user: {deleted_user['name']}")
else:
    print("No user found to delete")

In [ ]:
# Delete all documents in collection (but keep collection)
# result = users.delete_many({})

# Drop entire collection (faster for large collections)
# users.drop()

print("Collection clearing examples (commented out for safety)")

# 7. Find and Modify Operations

In [ ]:
# find_one_and_update - Update and return document

# Return original document (before update)
user = users.find_one_and_update(
    {"name": "John Doe"},
    {"$inc": {"age": 1}}
)
print(f"Before update - age: {user['age']}")

# Return modified document (after update)
from pymongo import ReturnDocument

user = users.find_one_and_update(
    {"name": "John Doe"},
    {"$inc": {"age": 1}},
    return_document=ReturnDocument.AFTER
)
print(f"After update - age: {user['age']}")

In [ ]:
# find_one_and_replace - Replace entire document

user = users.find_one_and_replace(
    {"_id": "user_eva_001"},
    {
        "_id": "user_eva_001",
        "name": "Eva Martinez",
        "email": "eva.new@email.com",
        "age": 33,
        "department": "Engineering"
    },
    return_document=ReturnDocument.AFTER
)

show(user, "Eva after replacement")

# 8. Complete CRUD Class

In [ ]:
from pymongo import MongoClient, ReturnDocument
from bson.objectid import ObjectId

class MongoCRUD:
    """Generic CRUD operations for MongoDB."""
    
    def __init__(self, uri, database, collection):
        self.client = MongoClient(uri)
        self.db = self.client[database]
        self.collection = self.db[collection]
    
    # CREATE
    def create(self, document):
        """Insert a single document."""
        result = self.collection.insert_one(document)
        return str(result.inserted_id)
    
    def create_many(self, documents):
        """Insert multiple documents."""
        result = self.collection.insert_many(documents)
        return [str(id) for id in result.inserted_ids]
    
    # READ
    def find_by_id(self, id):
        """Find document by _id."""
        if isinstance(id, str) and ObjectId.is_valid(id):
            id = ObjectId(id)
        return self.collection.find_one({"_id": id})
    
    def find_one(self, filter={}):
        """Find first matching document."""
        return self.collection.find_one(filter)
    
    def find_all(self, filter={}, sort=None, limit=0, skip=0):
        """Find all matching documents."""
        cursor = self.collection.find(filter)
        if sort:
            cursor = cursor.sort(sort)
        if skip:
            cursor = cursor.skip(skip)
        if limit:
            cursor = cursor.limit(limit)
        return list(cursor)
    
    def count(self, filter={}):
        """Count matching documents."""
        return self.collection.count_documents(filter)
    
    # UPDATE
    def update_by_id(self, id, update):
        """Update document by _id."""
        if isinstance(id, str) and ObjectId.is_valid(id):
            id = ObjectId(id)
        result = self.collection.update_one(
            {"_id": id},
            {"$set": update}
        )
        return result.modified_count
    
    def update_many(self, filter, update):
        """Update all matching documents."""
        result = self.collection.update_many(filter, {"$set": update})
        return result.modified_count
    
    # DELETE
    def delete_by_id(self, id):
        """Delete document by _id."""
        if isinstance(id, str) and ObjectId.is_valid(id):
            id = ObjectId(id)
        result = self.collection.delete_one({"_id": id})
        return result.deleted_count
    
    def delete_many(self, filter):
        """Delete all matching documents."""
        result = self.collection.delete_many(filter)
        return result.deleted_count
    
    def close(self):
        """Close the connection."""
        self.client.close()


# Usage example:
# crud = MongoCRUD("mongodb://localhost:27017/", "myapp", "users")
# id = crud.create({"name": "Test", "email": "test@email.com"})
# user = crud.find_by_id(id)
# crud.update_by_id(id, {"name": "Updated"})
# crud.delete_by_id(id)
# crud.close()

print("MongoCRUD class defined!")

# 9. Summary

## CRUD Operations Quick Reference

| Operation | Method | Returns |
|-----------|--------|--------|
| Insert one | `insert_one(doc)` | InsertOneResult (inserted_id) |
| Insert many | `insert_many([docs])` | InsertManyResult (inserted_ids) |
| Find one | `find_one(filter)` | Document or None |
| Find many | `find(filter)` | Cursor |
| Update one | `update_one(filter, update)` | UpdateResult |
| Update many | `update_many(filter, update)` | UpdateResult |
| Delete one | `delete_one(filter)` | DeleteResult |
| Delete many | `delete_many(filter)` | DeleteResult |

## Common Query Operators

| Operator | Meaning |
|----------|---------|
| `$eq`, `$ne` | Equal, Not equal |
| `$gt`, `$gte` | Greater than, Greater or equal |
| `$lt`, `$lte` | Less than, Less or equal |
| `$in`, `$nin` | In array, Not in array |
| `$and`, `$or` | Logical AND, OR |
| `$exists` | Field exists |
| `$regex` | Regular expression |

## Common Update Operators

| Operator | Description |
|----------|-------------|
| `$set` | Set field value |
| `$unset` | Remove field |
| `$inc` | Increment number |
| `$push` | Add to array |
| `$pull` | Remove from array |
| `$addToSet` | Add unique to array |

In [ ]:
# Cleanup
client.close()
print("Connection closed.")